**2) Duplicates & Irrelevant Columns**

This notebook identifies duplicate records, duplicate disease-symptom combinations, constant columns, near-constant columns, and irrelevant columns in the healthcare disease prediction dataset.

### Objective

- Identify exact duplicate records.
- Identify duplicate disease-symptom combinations.
- Check columns for unnecessary or irrelevant information.
- Detect constant and near-constant columns.
- Analyze the disease target variable.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import csv
from io import StringIO

### Load Dataset

In [ ]:
url = "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/main/healthcare-disease-prediction/dataset/healthcare_dataset.csv"

# Download CSV
response = requests.get(url)
response.raise_for_status()

# Read CSV without pandas' strict parser
reader = csv.reader(StringIO(response.text))
rows = list(reader)

# Separate header and data
header = rows[0]
data_rows = rows[1:]

# Find the maximum number of columns
max_fields = max(len(row) for row in data_rows)

# Add column names if extra fields exist
while len(header) < max_fields:
    header.append(f"Symptom_{len(header)}")

# Make all rows the same length
fixed_rows = []

for row in data_rows:
    if len(row) < max_fields:
        row = row + [""] * (max_fields - len(row))
    elif len(row) > max_fields:
        row = row[:max_fields]

    fixed_rows.append(row)

# Create DataFrame
df = pd.DataFrame(fixed_rows, columns=header)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

### 1. Check Exact Duplicate Records

In [ ]:
duplicate_count = df.duplicated().sum()

print("Number of exact duplicate records:", duplicate_count)

In [ ]:
duplicate_percentage = (duplicate_count / len(df)) * 100

print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

### Display Duplicate Records

In [ ]:
duplicates = df[df.duplicated(keep=False)]

print("Duplicate records:")
duplicates.head(20)

### 2. Identify Disease and Symptom Columns

In [ ]:
disease_column = "Disease"
symptom_columns = [col for col in df.columns if col.startswith("Symptom_")]

print("Disease column:", disease_column)
print("\nSymptom columns:")
print(symptom_columns)
print("\nNumber of symptom columns:", len(symptom_columns))

### 3. Check Duplicate Disease-Symptom Combinations

Two records are considered duplicates when the disease and all available symptom fields are identical.

In [ ]:
combination_columns = [disease_column] + symptom_columns

combination_duplicate_count = df.duplicated(
    subset=combination_columns
).sum()

print("Duplicate disease-symptom combinations:", combination_duplicate_count)

In [ ]:
combination_duplicate_percentage = (
    combination_duplicate_count / len(df)
) * 100

print(
    f"Duplicate disease-symptom combination percentage: "
    f"{combination_duplicate_percentage:.2f}%"
)

### 4. Column Analysis

In [ ]:
column_analysis = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Unique Values": [df[col].nunique(dropna=False) for col in df.columns],
    "Unique Percentage": [
        (df[col].nunique(dropna=False) / len(df)) * 100
        for col in df.columns
    ]
})

column_analysis

### 5. Identify Constant Columns

A constant column contains only one unique value and provides no useful information for machine learning.

In [ ]:
constant_columns = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 1
]

print("Constant columns:")
print(constant_columns)

### 6. Identify Near-Constant Columns

A near-constant column has the same value in at least 99% of the records.

In [ ]:
near_constant_columns = []

for col in df.columns:
    value_counts = df[col].value_counts(dropna=False)
    if len(value_counts) > 0:
        highest_percentage = (value_counts.iloc[0] / len(df)) * 100
        if highest_percentage >= 99:
            near_constant_columns.append((col, highest_percentage))

print("Near-constant columns (>=99% same value):")

if near_constant_columns:
    for col, percentage in near_constant_columns:
        print(f"{col}: {percentage:.2f}%")
else:
    print("No near-constant columns found.")

### 7. Identify Irrelevant Columns

In [ ]:
expected_columns = [disease_column] + symptom_columns

irrelevant_columns = [
    col for col in df.columns
    if col not in expected_columns
]

print("Columns outside the expected Disease + Symptom features:")
print(irrelevant_columns)

### Feature Relevance

| Column Type | Role | Relevance |
|---|---|---|
| Disease | Target variable | Required for prediction |
| Symptom_1 to Symptom_n | Input features | Required for prediction |
| Constant columns | No variation | Can be removed |
| Irrelevant columns | No predictive meaning | Can be removed |

### 8. Analyze Target Variable — Disease

In [ ]:
print("Number of unique diseases:", df[disease_column].nunique())

print("\nDisease names:")
print(df[disease_column].unique())

In [ ]:
disease_frequency = df[disease_column].value_counts().reset_index()
disease_frequency.columns = ["Disease", "Record_Count"]

disease_frequency

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(
    data=df,
    y=disease_column,
    order=df[disease_column].value_counts().index
)
plt.title("Number of Records for Each Disease")
plt.xlabel("Number of Records")
plt.ylabel("Disease")
plt.tight_layout()
plt.show()

### 9. Duplicate Records by Disease

In [ ]:
disease_duplicate_summary = df.groupby(disease_column).size().reset_index(name="Record_Count")
disease_duplicate_summary = disease_duplicate_summary.sort_values(
    by="Record_Count",
    ascending=False
)

disease_duplicate_summary.head(20)

### 10. Final Summary

In [ ]:
print("=" * 55)
print("DUPLICATE & IRRELEVANT COLUMN ANALYSIS SUMMARY")
print("=" * 55)
print(f"Total records              : {len(df)}")
print(f"Total columns              : {len(df.columns)}")
print(f"Number of symptom columns  : {len(symptom_columns)}")
print(f"Exact duplicate records    : {duplicate_count}")
print(f"Duplicate percentage       : {duplicate_percentage:.2f}%")
print(f"Disease-symptom duplicates : {combination_duplicate_count}")
print(f"Unique diseases            : {df[disease_column].nunique()}")
print(f"Constant columns           : {len(constant_columns)}")
print(f"Irrelevant columns         : {len(irrelevant_columns)}")
print("=" * 55)

### Conclusion

The dataset was checked for exact duplicate records, duplicate disease-symptom combinations, constant columns, near-constant columns, and irrelevant columns. These checks help identify unnecessary or repeated information before further preprocessing and machine learning.